# Exercise 6: Harmonic model

This exercise explores fundamental frequency ($f_0$) estimation and the harmonic model using `smstools`. It has four parts:
1. Estimate $f_0$ of one source in a polyphonic audio signal
2. Segment stable note regions using $f_0$
3. Measure inharmonicity in a pitched sound
4. [Optional] Improve the Two-Way Mismatch (TWM) $f_0$ estimation algorithm

### Relevant concepts

**Harmonic model parameters:** The harmonic model extends the sinusoidal model by constraining the analysis to integer multiples of a fundamental frequency. Beyond the standard STFT parameters, the key additional parameters are:

- `nH`: maximum number of harmonics per frame
- `minf0` / `maxf0`: frequency range (Hz) for $f_0$ candidates — narrowing this improves accuracy and speed
- `f0et`: error threshold for the TWM algorithm; if the mismatch error exceeds this, $f_0 = 0$ is returned for that frame
- `harmDevSlope`: slope controlling how much harmonic deviation from perfect integer multiples is tolerated at higher harmonics

**Melody representation in cents:** For musically meaningful analysis, $f_0$ is converted from Hz to cents using A1 = 55 Hz as the reference (0 cents):

$$f_{0,\mathrm{cents}} = 1200\log_2\!\left(\frac{f_{0,\mathrm{Hz}}}{55}\right)$$

**Segmentation:** A simple approach to note-level melody segmentation computes the standard deviation of $f_0$ (in cents) over a short sliding window. Regions where the standard deviation stays below a threshold `stdThsld` for at least `minNoteDur` seconds are classified as stable notes.

**Inharmonicity:** Real instruments deviate from perfect harmonicity — their partials are not exactly at integer multiples of $f_0$. Fletcher's model for piano strings gives the frequency of the $r$th partial as:

$$f_r = r\,f_0\sqrt{1 + Br^2}$$

A frame-level inharmonicity measure is:

$$I[l] = \frac{1}{R}\sum_{r=1}^{R}\frac{\left|f_{\mathrm{est}}^r[l] - r\,f_0[l]\right|}{r}$$

where $R$ is the number of detected harmonics in that frame (use only non-zero harmonics). The mean inharmonicity over frames $l_1$ to $l_2$ is:

$$I_{\mathrm{mean}} = \frac{1}{l_2 - l_1 + 1}\sum_{l=l_1}^{l_2} I[l]$$

**TWM $f_0$ candidates:** The Two-Way Mismatch algorithm tests each detected spectral peak as a potential $f_0$. This fails when the fundamental is absent from the spectrum (e.g. low-pitched voices). One improvement is to also include sub-harmonics of detected peaks as additional $f_0$ candidates. A second limitation is the assumption of perfect harmonicity, which degrades performance on piano sounds. Use `np.where(condition)` to efficiently index into numpy arrays.


## Part 1 – Estimate $f_0$ in a polyphonic audio signal

Tune the parameters of `f0Detection()` to track the melodic line in `cello-double-2.wav` — a recording of two cello strings played simultaneously. One string holds a constant drone; the other plays a simple melody. Your task is to track only the melody.

**Parameters to set:** `window`, `M`, `N`, `f0et`, `t`, `minf0`, `maxf0`. The hop size `H = 256` is fixed.

**Guidelines:**
- Use `minf0` / `maxf0` to exclude the drone frequency from the candidate range.
- `M` must be large enough to resolve spectral peaks belonging to different strings, but small enough to preserve note transitions without temporal smearing.
- `f0et` controls how strictly the TWM error is thresholded — too tight and frames will be dropped; too loose and wrong candidates are accepted.

> Do not use trial and error. Listen to the file, identify the approximate melody range, then set parameters analytically. Iterate from that starting point.


In [ ]:
import numpy as np
import math
from scipy.signal import get_window
import matplotlib.pyplot as plt
import IPython.display as ipd

from smstools.models import utilFunctions as UF
from smstools.models import harmonicModel as HM
from smstools.models import sineModel as SM
from smstools.models import stft
from smstools.models import dftModel as DFT

eps = np.finfo(float).eps

In [ ]:
# E6 - 1.1: Set parameters below, then run to perform f0 detection and visualise the result.

input_file = '../sounds/cello-double-2.wav'

### Set these analysis parameters
window = 'XX'
M = XX
N = XX
f0et = XX
t = XX
minf0 = XX
maxf0 = XX

# no need to modify below
H = 256
fs, x = UF.wavread(input_file)
w  = get_window(window, M)
f0 = HM.f0Detection(x, fs, w, N, H, t, minf0, maxf0, f0et)
y = UF.sinewaveSynth(f0, 0.8, H, fs)

ipd.display(ipd.Audio(data=x, rate=fs))
ipd.display(ipd.Audio(data=y, rate=fs))

maxplotfreq = 500.0
fig = plt.figure(figsize=(15, 9))

mX, pX = stft.stftAnal(x, w, N, H)
mX = np.transpose(mX[:,:int(N*(maxplotfreq/fs))+1])

timeStamps = np.arange(mX.shape[1])*H/float(fs)
binFreqs = np.arange(mX.shape[0])*fs/float(N)

plt.pcolormesh(timeStamps, binFreqs, mX, shading='auto')
plt.plot(timeStamps, f0, color='k', linewidth=1.5)
plt.ylabel('Frequency (Hz)')
plt.xlabel('Time (s)')
plt.legend(('f0',))
plt.tight_layout()
plt.show()


**E6 - 1.3: Conceptual questions (answer here)**

1. Report the parameter values you chose (`window`, `M`, `minf0`, `maxf0`, `f0et`, `t`). For each one, explain *why* you chose that value — what aspect of the signal or the algorithm motivated it?
2. Look at the $f_0$ contour plotted over the spectrogram. Are there frames where $f_0 = 0$ (detection failure)? At what moments do they occur, and what causes them?
3. Listen to the synthesised $f_0$ signal and compare it with the original. Can you identify note transitions correctly? Are there any frames where the drone pitch is accidentally tracked instead of the melody? What parameter change would help?


## Part 2 – Segment stable note regions using $f_0$

Given the $f_0$ contour of a monophonic signal, identify the time regions where the pitch is stable — these correspond to individual notes of the melody. The function `segment_stable_frequency_regions()` is already implemented below.

**Parameters to set:** `input_file`, `stdThsld`, `minNoteDur`, `winStable`, plus the harmonic analysis parameters `window`, `M`, `N`, `H`, `f0et`, `t`, `minf0`, `maxf0`.

Use `sax-phrase-short.wav` as the primary test signal. Choose parameters that produce the most musically meaningful note segmentation — one segment per clearly played note. Synthesise the detected $f_0$ segments as sinusoids and compare with the original audio.


In [ ]:
# function used in exercise

def segment_stable_frequency_regions(f0, stdThsld, minNoteDur, winStable):
    """Segment the stable regions of a fundamental frequency track.

    Args:
        f0 (np.array): f0 values of a sound
        stdThsld (float): threshold for detecting stable regions in the f0 contour (in cents)
        minNoteDur (float): minimum allowed segment length (note duration)
        winStable (int): number of samples used for computing standard deviation

    Result:
        segments (np.array): starting and ending frame indexes of every segment

    """

    # convert f0 values from Hz to Cents (as described in pdf document)
    f0Cents = 1200*np.log2((f0+eps)/55.0)

    # create an array containing standard deviation of last winStable samples
    stdArr = 10000000000*np.ones(f0.shape)
    for ii in range(winStable-1, len(f0)):
        stdArr[ii] = np.std(f0Cents[ii-winStable+1:ii+1])

    # apply threshold on standard deviation values to find indexes of the stable points in melody
    indFlat = np.where(stdArr<=stdThsld)[0]
    flatArr = np.zeros(f0.shape)
    flatArr[indFlat] = 1

    # create segments of continuous stable points such that consecutive stable points belong to same segment
    onset = np.where((flatArr[1:]-flatArr[:-1])==1)[0]+1
    offset = np.where((flatArr[1:]-flatArr[:-1])==-1)[0]

    # remove any offset before onset (to sync them)
    indRem = np.where(offset<onset[0])[0]
    offset = np.delete(offset, indRem)

    minN = min(onset.size, offset.size)
    segments = np.transpose(np.vstack((onset[:minN], offset[:minN])))

    # apply segment filtering, i.e. remove segments with are < minNoteDur in length
    minNoteSamples = int(np.ceil(minNoteDur*fs/H))
    diff = segments[:,1] - segments[:,0]
    indDel = np.where(diff<minNoteSamples)
    segments = np.delete(segments,indDel, axis=0)

    return segments

Test cases for `segment_stable_frequency_regions()`:

**Test case 1:** `cello-phrase.wav`, `stdThsld=10`, `minNoteDur=0.1`, `winStable=3`, `window='hamming'`, `M=1025`, `N=2048`, `H=256`, `f0et=5.0`, `t=-100`, `minf0=310`, `maxf0=650` → **9 segments**

**Test case 2:** `cello-phrase.wav`, `stdThsld=20`, `minNoteDur=0.5`, `winStable=3`, same analysis params → **6 segments**

**Test case 3:** `sax-phrase-short.wav`, `stdThsld=5`, `minNoteDur=0.6`, `winStable=3`, same analysis params → **1 segment**

These parameter sets are provided for code validation only — they are not necessarily optimal for musical note segmentation.


In [ ]:
# E6 - 2.1: Set parameters, compute f0 and segments, then plot spectrogram with f0 and segments.
#            Synthesise the f0 segments and compare with the original audio.

### Set these parameters
input_file = '../sounds/sax-phrase-short.wav'
stdThsld = XX
minNoteDur = XX
winStable = XX
window = 'XX'
M = XX
N = XX
H = XX
f0et = XX
t = XX
minf0 = XX
maxf0 = XX

# no need to change below
fs, x = UF.wavread(input_file)
w  = get_window(window, M)
f0 = HM.f0Detection(x, fs, w, N, H, t, minf0, maxf0, f0et)
segments = segment_stable_frequency_regions(f0, stdThsld, minNoteDur, winStable)

maxplotfreq = 1000.0
plt.figure(figsize=(15, 9))

mX, pX = stft.stftAnal(x, w, N, H)
mX = np.transpose(mX[:,:int(N*(maxplotfreq/fs))+1])

timeStamps = np.arange(mX.shape[1])*H/float(fs)
binFreqs = np.arange(mX.shape[0])*fs/float(N)

plt.pcolormesh(timeStamps, binFreqs, mX, shading='auto')
plt.plot(timeStamps, f0, color='k', linewidth=5)

for i in range(segments.shape[0]):
    plt.plot(timeStamps[segments[i,0]:segments[i,1]], f0[segments[i,0]:segments[i,1]],
             color='#A9E2F3', linewidth=1.5)
plt.ylabel('Frequency (Hz)', fontsize=12)
plt.xlabel('Time (s)', fontsize=12)
plt.legend(('f0', 'segments'))
plt.tight_layout()
plt.show()

print(f'Number of segments detected: {segments.shape[0]}')


**E6 - 2.3: Conceptual questions (answer here)**

1. How many segments did your parameter choices produce for `sax-phrase-short.wav`? Do they align with the notes you hear? If the segmentation under- or over-segments, which parameter (`stdThsld`, `minNoteDur`, `winStable`) would you adjust and in which direction?
2. Listen to the synthesised $f_0$ segments alongside the original. Are the pitch values accurate? If a segment's synthesised pitch sounds "off", what might explain the discrepancy (estimation error, vibrato, intonation drift)?
3. This segmentation approach fails for notes with vibrato. Explain why the standard deviation criterion breaks down for vibrato, and propose a modification that could handle it.


## Part 3 – Measure inharmonicity in a pitched sound

The function `estimate_inharmonicity()` should compute the mean inharmonicity of a sound from its harmonic frequencies obtained via `harmonicModelAnal()`.

**Implementation notes:**
- Use the frame-level formula from the Relevant concepts section.
- Some harmonics may be undetected (zero frequency) in certain frames — use only the non-zero harmonics to compute $I[l]$, setting $R$ to the count of detected harmonics in that frame.
- Average $I[l]$ over the frames of interest to obtain $I_\mathrm{mean}$.

**Input:** `xhfreq` — 2D array of shape `(num_frames, nH)` containing harmonic frequencies in Hz (zeros for undetected harmonics).

**Output:** `(float)` — mean inharmonicity over all frames.


In [ ]:
# E6 - 3.1: 
# Complete function estimate_inharmonicity()

def estimate_inharmonicity(xhfreq):
    """Estimate inharmonicity factor from the psedo-harmonic frequencies of a sound.

    Args:
        xhfreq (np.array): harmonic frequencies of a sound

    Output:
        (float): mean inharmonicity over all the frames

    """
    ### Your code here


Test cases for `estimate_inharmonicity()` — first run `harmonicModelAnal()` with the parameters below, then pass the harmonic frequency track to `estimate_inharmonicity()`. The time boundaries `t1` and `t2` select the relevant frames from the full analysis.

**Test case 1:** `piano.wav`, `t1=0.2`, `t2=0.4`, `window='hamming'`, `M=2047`, `N=2048`, `H=128`, `f0et=5.0`, `t=-90`, `minf0=130`, `maxf0=180`, `nH=25` → expected output: `1.4607`

**Test case 2:** `piano.wav`, `t1=2.3`, `t2=2.55`, same analysis params, `minf0=230`, `maxf0=290`, `nH=15` → expected output: `1.4852`

**Test case 3:** `piano.wav`, `t1=2.55`, `t2=2.8`, same analysis params, `minf0=230`, `maxf0=290`, `nH=5` → expected output: `0.1743`

After validating, compare inharmonicity values across different instruments (e.g., violin vs. piano).


In [ ]:
# E6 - 3.2: Run harmonicModelAnal() on piano.wav for the test cases above, call estimate_inharmonicity(),
#            and plot the harmonic frequency tracks used in the computation.

### Your code here


**E6 - 3.3: Conceptual questions (answer here)**

1. Report the mean inharmonicity values for the three test cases. How do they compare to each other? Does the number of harmonics `nH` affect the result (compare test cases 2 and 3, which cover the same note)?
2. Why does greater inharmonicity appear at higher harmonics? Refer to Fletcher's model — how does the deviation $f_r - r\,f_0$ grow with the harmonic number $r$?
3. Compare inharmonicity of the piano with another instrument of your choice (e.g., violin or flute). Based on the physical structure of that instrument, would you expect more or less inharmonicity than the piano? Verify empirically.


## [Optional] Part 4 – Improve the TWM $f_0$ estimation algorithm

This part asks you to identify and address specific failure modes of the TWM algorithm by modifying the local copies of `f0Detection()`, `f0Twm()`, and `TWM_p()` provided below. All changes should be made only to the notebook cells — do not modify the `smstools` package.

**Two known failure scenarios:**

1. **Missing fundamental:** If the $f_0$ partial is too weak to be detected as a spectral peak, the current implementation has no candidate for the true $f_0$. Improvement: also add sub-harmonics of detected peaks (e.g. $f_\mathrm{peak}/2$, $f_\mathrm{peak}/3$, …) as additional $f_0$ candidates in `f0Twm()`.

2. **Piano inharmonicity:** `TWM_p()` generates ideal harmonic series at exact integer multiples. For piano, the partials are slightly stretched (see Fletcher's formula). Improvement: generate the candidate harmonic series using $f_r = r\,f_0\sqrt{1 + Br^2}$ with a small empirical $B$ value instead of exact multiples.

**Workflow:**
1. First verify that the best achievable performance with the current parameters (cell below) is already limited.
2. Make targeted changes to the copies of the functions below.
3. Plot the improved $f_0$ contour on top of the spectrogram and compare with the original output.

> Start with the best parameter tuning before modifying the algorithm — parameter errors should not be confused with algorithmic limitations.


In [ ]:
### modify anything in f0Detection()
def f0Detection(x, fs, w, N, H, t, minf0, maxf0, f0et):
    """
    Fundamental frequency detection of a sound using twm algorithm
    x: input sound; fs: sampling rate; w: analysis window;
    N: FFT size; t: threshold in negative dB,
    minf0: minimum f0 frequency in Hz, maxf0: maximim f0 frequency in Hz,
    f0et: error threshold in the f0 detection (ex: 5),
    returns f0: fundamental frequency
    """
    if (minf0 < 0):                                            # raise exception if minf0 is smaller than 0
        raise ValueError("Minumum fundamental frequency (minf0) smaller than 0")

    if (maxf0 >= 10000):                                       # raise exception if maxf0 is bigger than fs/2
        raise ValueError("Maximum fundamental frequency (maxf0) bigger than 10000Hz")

    if (H <= 0):                                               # raise error if hop size 0 or negative
        raise ValueError("Hop size (H) smaller or equal to 0")

    hN = N/2                                                   # size of positive spectrum
    hM1 = int(math.floor((w.size+1)/2))                        # half analysis window size by rounding
    hM2 = int(math.floor(w.size/2))                            # half analysis window size by floor
    x = np.append(np.zeros(hM2),x)                             # add zeros at beginning to center first window at sample 0
    x = np.append(x,np.zeros(hM1))                             # add zeros at the end to analyze last sample
    pin = hM1                                                  # init sound pointer in middle of anal window
    pend = x.size - hM1                                        # last sample to start a frame
    fftbuffer = np.zeros(N)                                    # initialize buffer for FFT
    w = w / sum(w)                                             # normalize analysis window
    f0 = []                                                    # initialize f0 output
    f0t = 0                                                    # initialize f0 track
    f0stable = 0                                               # initialize f0 stable
    while pin<pend:
        x1 = x[pin-hM1:pin+hM2]                                  # select frame
        mX, pX = DFT.dftAnal(x1, w, N)                           # compute dft
        ploc = UF.peakDetection(mX, t)                           # detect peak locations
        iploc, ipmag, ipphase = UF.peakInterp(mX, pX, ploc)      # refine peak values
        ipfreq = fs * iploc/N                                    # convert locations to Hez
        f0t = f0Twm(ipfreq, ipmag, f0et, minf0, maxf0, f0stable)  # find f0
        if ((f0stable==0)&(f0t>0)) \
                or ((f0stable>0)&(np.abs(f0stable-f0t)<f0stable/5.0)):
            f0stable = f0t                                         # consider a stable f0 if it is close to the previous one
        else:
            f0stable = 0
        f0 = np.append(f0, f0t)                                  # add f0 to output array
        pin += H                                                 # advance sound pointer
    return f0


In [ ]:
### modify anything in f0Twm()
def f0Twm(pfreq, pmag, ef0max, minf0, maxf0, f0t=0):
    """
    Function that wraps the f0 detection function TWM, selecting the possible f0 candidates
    and calling the function TWM with them
    pfreq, pmag: peak frequencies and magnitudes,
    ef0max: maximum error allowed, minf0, maxf0: minimum  and maximum f0
    f0t: f0 of previous frame if stable
    returns f0: fundamental frequency in Hz
    """
    if (minf0 < 0):                                  # raise exception if minf0 is smaller than 0
        raise ValueError("Minumum fundamental frequency (minf0) smaller than 0")

    if (maxf0 >= 10000):                             # raise exception if maxf0 is bigger than 10000Hz
        raise ValueError("Maximum fundamental frequency (maxf0) bigger than 10000Hz")

    if (pfreq.size < 3) & (f0t == 0):                # return 0 if less than 3 peaks and not previous f0
        return 0

    f0c = np.argwhere((pfreq>minf0) & (pfreq<maxf0))[:,0] # use only peaks within given range
    if (f0c.size == 0):                              # return 0 if no peaks within range
        return 0
    f0cf = pfreq[f0c]                                # frequencies of peak candidates
    f0cm = pmag[f0c]                                 # magnitude of peak candidates

    if f0t>0:                                        # if stable f0 in previous frame
        shortlist = np.argwhere(np.abs(f0cf-f0t)<f0t/2.0)[:,0]   # use only peaks close to it
        maxc = np.argmax(f0cm)
        maxcfd = f0cf[maxc]%f0t
        if maxcfd > f0t/2:
            maxcfd = f0t - maxcfd
        if (maxc not in shortlist) and (maxcfd>(f0t/4)): # or the maximum magnitude peak is not a harmonic
            shortlist = np.append(maxc, shortlist)
        f0cf = f0cf[shortlist]                         # frequencies of candidates

    if (f0cf.size == 0):                             # return 0 if no peak candidates
        return 0

    f0, f0error = TWM_p(pfreq, pmag, f0cf)        # call the TWM function with peak candidates

    if (f0>0) and (f0error<ef0max):                  # accept and return f0 if below max error allowed
        return f0
    else:
        return 0


In [ ]:
### modify anything in TWM_p()
def TWM_p(pfreq, pmag, f0c):
    """
    Two-way mismatch algorithm for f0 detection (by Beauchamp&Maher)
    [better to use the C version of this function: UF_C.twm]
    pfreq, pmag: peak frequencies in Hz and magnitudes,
    f0c: frequencies of f0 candidates
    returns f0, f0Error: fundamental frequency detected and its error
    """

    p = 0.5                                          # weighting by frequency value
    q = 1.4                                          # weighting related to magnitude of peaks
    r = 0.5                                          # scaling related to magnitude of peaks
    rho = 0.33                                       # weighting of MP error
    Amax = max(pmag)                                 # maximum peak magnitude
    maxnpeaks = 10                                   # maximum number of peaks used
    harmonic = np.matrix(f0c)
    ErrorPM = np.zeros(harmonic.size)                # initialize PM errors
    MaxNPM = min(maxnpeaks, pfreq.size)
    for i in range(0, MaxNPM) :                      # predicted to measured mismatch error
        difmatrixPM = harmonic.T * np.ones(pfreq.size)
        difmatrixPM = abs(difmatrixPM - np.ones((harmonic.size, 1))*pfreq)
        FreqDistance = np.amin(difmatrixPM, axis=1)    # minimum along rows
        peakloc = np.argmin(difmatrixPM, axis=1)
        Ponddif = np.array(FreqDistance) * (np.array(harmonic.T)**(-p))
        PeakMag = pmag[peakloc]
        MagFactor = 10**((PeakMag-Amax)/20)
        ErrorPM = ErrorPM + (Ponddif + MagFactor*(q*Ponddif-r)).T
        harmonic = harmonic+f0c

    ErrorMP = np.zeros(harmonic.size)                # initialize MP errors
    MaxNMP = min(maxnpeaks, pfreq.size)
    for i in range(0, f0c.size) :                    # measured to predicted mismatch error
        nharm = np.round(pfreq[:MaxNMP]/f0c[i])
        nharm = (nharm>=1)*nharm + (nharm<1)
        FreqDistance = abs(pfreq[:MaxNMP] - nharm*f0c[i])
        Ponddif = FreqDistance * (pfreq[:MaxNMP]**(-p))
        PeakMag = pmag[:MaxNMP]
        MagFactor = 10**((PeakMag-Amax)/20)
        ErrorMP[i] = sum(MagFactor * (Ponddif + MagFactor*(q*Ponddif-r)))

    Error = (ErrorPM[0]/MaxNPM) + (rho*ErrorMP/MaxNMP)  # total error
    f0index = np.argmin(Error)                       # get the smallest error
    f0 = f0c[f0index]                                # f0 with the smallest error

    return f0, Error[f0index]

In [ ]:
# E6 - 4.1: 
# Modify what you thing might import the fo detection of the functions
# f0Twm(), TWM_p() and f0Detection(). Plot the f0 contour on top of the spectrogram
# of the original sound and play the f0 function. Explain what you tried to do
# and what results did you obtained.

### modify anything

input_file = '../sounds/piano.wav'
window = 'hamming'
M = 2048
N = 2048
H = 256
f0et = 5.0
t = -80
minf0 = 100
maxf0 = 300

ipd.display(ipd.Audio('../sounds/piano.wav'))
fs, x = UF.wavread(input_file)
w  = get_window(window, M)
f0 = f0Detection(x, fs, w, N, H, t, minf0, maxf0, f0et)

## Code for plotting the f0 contour on top of the spectrogram
maxplotfreq = 500.0
plt.figure(figsize=(15, 5))

mX, pX = stft.stftAnal(x, w, N, H)
mX = np.transpose(mX[:,:int(N*(maxplotfreq/fs))+1])

timeStamps = np.arange(mX.shape[1])*H/float(fs)
binFreqs = np.arange(mX.shape[0])*fs/float(N)

plt.pcolormesh(timeStamps, binFreqs, mX, shading='auto')
plt.plot(timeStamps, f0, color = 'k', linewidth=1.5)

plt.ylabel('Frequency (Hz)')
plt.xlabel('Time (s)')

**E6 - 4.3: Conceptual questions (answer here)**

1. Describe the specific failure mode you tried to fix (missing fundamental or piano inharmonicity). Show the $f_0$ contour before and after your modification. Was the improvement measurable?
2. For the missing-fundamental case: explain where in `f0Twm()` you added sub-harmonic candidates. What is the risk of adding too many sub-harmonic levels?
3. For the piano case: what value of $B$ did you use in Fletcher's formula? How did you choose it — empirically or from the literature? How sensitive is the result to the exact value of $B$?
4. Are the two proposed improvements compatible — could you apply both simultaneously? Would doing so help or hurt for a signal that is neither a piano nor a low-pitched voice?
